# Analyse exploratoire longitudinale

**Objectif :** étudier les associations entre les caractéristiques des inscriptions et l’abandon dans les 30 jours suivant les paliers J30, J60 et J90 (`target_next_30d`).

Le panel code les moyennes manquantes à zéro. Les observations répétées d’un même étudiant ne sont pas indépendantes ; les tests statistiques ont donc une portée exploratoire.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Ouvrir ce notebook depuis la racine du projet ou notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.config import LONGITUDINAL_PANEL
from scipy.stats import mannwhitneyu, chi2_contingency, false_discovery_control
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

## Préparation des données

### Chargement du panel

In [ ]:
data = pd.read_parquet(LONGITUDINAL_PANEL)
print(f"Observations longitudinales : {len(data)}")

### Exclusion des identifiants et des variables non utilisées

In [ ]:
redundant_cols = [
    'id_inscription', 'id_etudiant', 'id_programme',
    'resultat_final', 'date_debut', 'date_annulation',
    'date_debut_programme', 'cancel_day', 'churn'
]

data.drop(columns=redundant_cols, inplace=True, errors='ignore')
print(f"Variables conservées : {len(data.columns)}")

### Types de variables

In [ ]:
num = [
    'age',
    'nb_inscriptions_precedentes',
    'volume_per_active_day_30d',
    'days_since_last_activity',
    'active_days_diff',
    'n_evaluations_30d',
    'avg_note_cum',
    'avg_lateness_cum'
]

cat = [
    'niveau_scolaire',
    'annee_scolaire',
    'semestre',
    'genre',
    'region',
    'situation_handicap'
]

target_column = 'target_next_30d'

## Visualisation des données

In [ ]:
print(data[target_column].value_counts(normalize=True))

print(
    pd.crosstab(
        data['checkpoint_day'],
        data[target_column],
        normalize='index'
    )
)

sns.barplot(data=data, x='checkpoint_day', y=target_column, errorbar=None)
plt.ylabel("Taux d'abandon dans les 30 jours")
plt.show()

### Variables numériques

In [ ]:
data[num].describe()

In [ ]:
data[num].isna().mean().sort_values(ascending=False)

In [ ]:
for col in num:
    sns.boxplot(data=data, x=target_column, y=col, showfliers=False)
    plt.title(col)
    plt.show()

### Variables catégorielles

In [ ]:
for col in cat:
    print(f'Pour la variable {col}:')
    print(data[col].value_counts())
    print(pd.crosstab(data[col],data[target_column],normalize='index'))
    print('\n')

In [ ]:
base_rate = data[target_column].mean()

for col in cat:
    churn_rate = data.groupby(col, observed=True)[target_column].mean().sort_values(ascending=False)

    sns.barplot(x=churn_rate.values, y=churn_rate.index.astype(str), orient='h', color='steelblue')
    plt.axvline(base_rate, color='red', linestyle='--', label=f'Taux global ({base_rate:.1%})')
    plt.xlabel("Taux d'abandon")
    plt.ylabel(col)
    plt.title(f"Taux d'abandon par {col}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Associations statistiques

### Test de Mann-Whitney pour les variables numériques

- $H_0$ : les distributions de la variable sont identiques dans les deux groupes.
- $H_1$ : les distributions diffèrent.

Le score $A=U/(n_0n_1)$ compare les rangs du groupe sans abandon dans les 30 jours au groupe avec abandon. Une valeur de 0,5 correspond à une absence de différence moyenne de rangs.

In [ ]:
res = []

for col in num:
    group0 = data.loc[data[target_column]==0, col].dropna()
    group1 = data.loc[data[target_column]==1, col].dropna()
    n0, n1 = len(group0), len(group1)

    U, p = mannwhitneyu(group0, group1, alternative='two-sided')
    score = U / (n0*n1)

    res.append({
        'variable': col,
        'test': 'Mann-Whitney',
        'statistique': U,
        'p_value': p,
        'score': score
    })

num_results = pd.DataFrame(res)
num_results

### Test du chi-deux pour les variables catégorielles

- $H_0$ : la variable et la cible sont indépendantes.
- $H_1$ : la variable et la cible sont associées.

Le V de Cramér mesure l’intensité de l’association.

In [ ]:
def cramers_v(contingency_table: pd.DataFrame) -> tuple[float, float, float]:
    """Association descriptive entre deux variables catégorielles."""
    if min(contingency_table.shape) < 2 or contingency_table.to_numpy().sum() == 0:
        return np.nan, np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(contingency_table)
    n = contingency_table.values.sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt(chi2 / (n*min_dim))
    return chi2, p, v

cat_res = []

for col in cat:
    table = pd.crosstab(data[col], data[target_column])
    chi2, p, v = cramers_v(table)

    cat_res.append({
        'variable': col,
        'test': 'Chi-2',
        'statistique': chi2,
        'p_value': p,
        'score': v
    })

cat_results = pd.DataFrame(cat_res)
cat_results

### Correction de Benjamini-Hochberg

In [ ]:
results = pd.concat([num_results, cat_results], ignore_index=True)
results['p_value_adj'] = false_discovery_control(results['p_value'], method='bh')
results.sort_values('p_value_adj')

## Corrélations

### Variables numériques

In [ ]:
corr = data[num].corr(method='spearman')
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.show()

### Variables catégorielles

In [ ]:
cramers_matrix = pd.DataFrame(index=cat, columns=cat, dtype=float)
for col1 in cat:
    for col2 in cat:
        table = pd.crosstab(data[col1], data[col2])
        _, _, v = cramers_v(table)
        cramers_matrix.loc[col1, col2] = v

sns.heatmap(cramers_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.show()

## Étude de la multicolinéarité

In [ ]:
vif_data = data[num].replace([np.inf, -np.inf], np.nan).dropna()
vif_data = vif_data.loc[:, vif_data.nunique() > 1]

vif_design = add_constant(vif_data.astype(float))

vif_results = pd.DataFrame({
    'variable': vif_design.columns,
    'VIF': [
        variance_inflation_factor(vif_design.values, i)
        for i in range(vif_design.shape[1])
    ]
})

vif_results = vif_results[vif_results['variable'] != 'const'].sort_values(
    'VIF', ascending=False
).reset_index(drop=True)

vif_results

## Conclusion

La récence de l’activité, la note moyenne et le niveau scolaire présentent des associations avec l’abandon à trente jours. Les associations du genre, de la région, du semestre et du nombre d’inscriptions précédentes sont faibles. Le handicap déclaré n’est pas retenu parmi les prédicteurs en raison de sa nature sensible.

Les observations répétées limitent l’interprétation des tests, même après correction des valeurs $p$. L’exploration porte sur l’ensemble du panel, y compris des élèves présents dans le test final.

Le faible nombre d’abandons dans certaines fenêtres et les performances prédictives insuffisantes ont conduit à abandonner l’approche longitudinale au profit d’une prédiction unique à J30 de l’abandon ultérieur enregistré.